# The DSPy Skill for CLI Generation

This notebook accompanies Chapter 10 §10.4.2 of *Context Engineering with DSPy*: The DSPy Skill for CLI Generation.

## Required environment variables

Add these to a `.env` at the repo root (see `.env.example`):

- `OPENAI_API_KEY` — used as the task / judge / router LM via `dspy.LM("openai/...")`.
- `OPENROUTER_API_KEY` — used when the notebook calls Anthropic models via OpenRouter (`dspy.LM("openrouter/anthropic/claude-opus-4.7")`). No Claude Code CLI is required.

## Original source

Imported from the writing repo at `working/cli-generation-skill.ipynb`.


In [ ]:
%pip install -r ../requirements.txt -q

# A DSPy skill that helps an agent build CLIs

When you ask a coding agent to "build a CLI that does X using DSPy," it usually hallucinates a 2023-era API -- `dspy.Module` with the wrong methods, `optimizer.compile` with the wrong arguments, signatures that don't validate. The fix isn't a better model, it's putting the *current* DSPy reference in front of the agent.

We've already got that reference: this book's `compacted-dspy-docs.md` is a single file with the entire current API surface. Wrapping it as a Claude Code skill is two lines of frontmatter. Then any agent that loads the skill writes DSPy that actually runs.

This notebook does three things:
1. Builds the `dspy` skill from `compacted-dspy-docs.md`.
2. Asks the same model to build the same CLI -- once without the skill, once with.
3. Sketches how you'd close the loop and *optimize the skill itself* using the wrap-as-module pattern from Nous Research's [hermes-agent-self-evolution](https://github.com/NousResearch/hermes-agent-self-evolution).

**Prerequisites:**
- `OPENAI_API_KEY` in your environment
- This notebook lives in `working/`; it reads `./assets/compacted-dspy-docs.md`

In [ ]:
import os, textwrap, dspy
from pathlib import Path

assert os.environ.get("OPENAI_API_KEY"), "set OPENAI_API_KEY first"

CODE_LM = dspy.LM("openai/gpt-5-mini", temperature=0.3, max_tokens=2000)

## Step 1: wrap the docs as a skill

Claude Code (and Cursor, Codex, etc.) treat any markdown file with YAML frontmatter as a dispatchable skill. The body is the actual instructions the agent reads; the `description` is what the parent agent reads when deciding whether to load it. Here we glue a four-line frontmatter onto the existing compacted docs:

In [ ]:
DOCS_PATH = Path("./assets/compacted-dspy-docs.md")
docs = DOCS_PATH.read_text()

DSPY_SKILL = textwrap.dedent("""\
    ---
    name: dspy
    description: Use when the user asks to build, modify, or debug a DSPy program (signatures, modules, optimizers, evaluation). Loads the current DSPy API reference into context.
    ---

    When asked to build a DSPy program:
    1. Define one `dspy.Signature` per stage. Use typed `dspy.InputField` / `dspy.OutputField`.
    2. Wrap them in a `dspy.Module` with `__init__` and `forward()`.
    3. Configure the LM once with `dspy.configure(lm=...)`.
    4. Don't invent APIs. If you're unsure, search this document first.

    Reference (the entire current DSPy surface):

    """) + docs

Path("./outputs").mkdir(exist_ok=True)
Path("./outputs/dspy_skill.md").write_text(DSPY_SKILL)
print(f"skill is {len(DSPY_SKILL):,} chars ({len(DSPY_SKILL.split()):,} words)")
print("first 8 lines (the frontmatter + opening rules):")
print("\n".join(DSPY_SKILL.splitlines()[:8]))

## Step 2: same task, with and without the skill

One concrete task: build a sentiment classifier as a DSPy CLI. We ask the same model the same question; the only difference is whether the skill is in its system prompt. The without-skill response is the baseline -- the model writes from memory, which usually means it invents API shapes that no longer exist. The with-skill response writes against the reference.

In [ ]:
TASK = (
    "Build a Python CLI that classifies a piece of user feedback as 'positive', "
    "'negative', or 'neutral' using DSPy. One file. Use the latest DSPy API. "
    "Output only the code, no commentary."
)

without_skill = CODE_LM(messages=[
    {"role": "user", "content": TASK},
])[0]

with_skill = CODE_LM(messages=[
    {"role": "system", "content": DSPY_SKILL},
    {"role": "user",   "content": TASK},
])[0]

Path("./outputs/cli_without_skill.py").write_text(without_skill)
Path("./outputs/cli_with_skill.py").write_text(with_skill)

print("WITHOUT skill (first 40 lines):")
print("\n".join(without_skill.splitlines()[:40]))
print("\n" + "=" * 60 + "\n")
print("WITH skill (first 40 lines):")
print("\n".join(with_skill.splitlines()[:40]))

## Step 3: does the code actually run?

The simplest possible metric: import the generated file and see if it loads. A file that imports things that don't exist (`dspy.PromptTemplate`, `dspy.Pipeline`) fails here; one that uses real DSPy primitives loads. That's the gskill metric pattern -- binary pass/fail from a deterministic check -- applied to code generation.

In [ ]:
import importlib.util, traceback, re

def loads_cleanly(path: str) -> tuple[bool, str]:
    src = Path(path).read_text()
    # Strip markdown code fences if the LM included them.
    src = re.sub(r"^```\w*\n|```$", "", src.strip(), flags=re.MULTILINE)
    Path(path).write_text(src)
    spec = importlib.util.spec_from_file_location("gen", path)
    module = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(module)
        return True, "loaded"
    except Exception:
        return False, traceback.format_exc(limit=2)

ok_no,  msg_no  = loads_cleanly("./outputs/cli_without_skill.py")
ok_yes, msg_yes = loads_cleanly("./outputs/cli_with_skill.py")

print(f"without skill: {'PASS' if ok_no  else 'FAIL'}")
if not ok_no:  print(msg_no.splitlines()[-1])
print(f"with skill:    {'PASS' if ok_yes else 'FAIL'}")
if not ok_yes: print(msg_yes.splitlines()[-1])

Save the skill to `~/.claude/skills/dspy/SKILL.md` (or your editor's equivalent) and every agent on your team that asks for a DSPy CLI gets one that runs on the first try. If you want to take this further -- evolving the opening rules against a benchmark of CLI briefs rather than hand-writing them -- Nous Research's [hermes-agent-self-evolution](https://github.com/NousResearch/hermes-agent-self-evolution) is the cleanest worked example. Worth reading before you point GEPA at your own production skills.